# 2/2 — Train 25 MB Mobile1D XL on segmented real + synthetic images

Run **Notebook 1 (synthesis and segmentation)** first. This notebook then checks and restores its SHA-verified Drive shards and trains Mobile1D XL on those **actual segmentation-derived 176×512 resistor-body crops**.

### Training defaults
- Model: 48 base channels, **568 feature channels**, 3 TCN blocks, 2 slot-reader layers. Estimated **6,333,985 parameters**, **25.34 MB decimal / 24.16 MiB FP32 weights** (the exported TFLite size is not guaranteed to be identical).
- 120 epochs, batch size 8 for T4, initial learning rate 0.00025 with 5 warmup epochs, cosine schedule, EMA and 5-epoch recovery snapshots.
- Real validation/test sources are not augmented; training sources include extra three-/four-/five-/six-band synthetic combinations. Horizontal flips are label-consistent at training time.
- The existing 1.4 MB model and earlier XL experiment are **not** resumed or overwritten. New checkpoints are isolated under `MyDrive/resistor_model/r1-mobile1d-xl-segsynthetic-colab/`.
- Uses the same source train/val/test assignment, but note the segmenter itself has a failure rate. Check the printed **real_source_coverage** on each split in addition to exact-value accuracy.
- Input is already crop-aligned, so no second whole-resistor rectification occurs. Export of ONNX and best-effort TFLite runs at the end.

### Operating costs
The model is ~17.2× the small model's parameters. GPU memory, training time, Android latency and real-world accuracy must be measured: larger does not automatically mean better. If Colab T4 runs out of memory, lower the XL batch size in `colab/train_rres_mobile1d_xl_colab.py` and start a new controlled experiment.


In [ ]:
import os,shutil,subprocess
from pathlib import Path
branch='feat/mobile1d-colab-progress-drive-cache'
os.environ['MOBILE1D_GIT_REF']=branch
repo=Path('/content/mobile1d_xl_segsynthetic_checkout')
if repo.exists(): shutil.rmtree(repo)
subprocess.run(['git','clone','--depth','1','--single-branch','--branch',branch,
    'https://github.com/Persie0/resistor_model.git',str(repo)],check=True)
version=subprocess.check_output(['git','-C',str(repo),'rev-parse','--short','HEAD'],text=True).strip()
script=repo/'colab/train_rres_mobile1d_xl_segsynthetic_colab.py'
source=script.read_bytes()
assert b'configure_segmented_training' in source
assert b'xl-segsynthetic' in source
print('[segsynthetic-XL] checked out',version)
exec(compile(source,str(script),'exec'),{'__name__':'__main__','__file__':str(script)})
